# Homework X2: MLP for MNIST Classification

**Optional; maximum 0 points; not counted toward the final grade.** Due October 26, 2026 (Monday), 12:00 noon.

Extend HW1's linear MNIST classifier to a one-hidden-layer MLP. Implement the five marked `TODO` blocks with NumPy; do not use a prebuilt neural network, loss, optimizer, or automatic differentiation. See `HWX2_MLP_MNIST.pdf` for the full assignment.

**Setup:** Python 3, NumPy, and Matplotlib. Keep this notebook, `mnist_data_loader.py`, and `mnist_data/` together. The local data need no download. Do not submit the dataset files.


In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# Also lets the instructor notebook run from its instructor/ subfolder.
for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd() / 'HWX2'):
    if (candidate / 'mnist_data_loader.py').is_file():
        sys.path.insert(0, str(candidate))
        break
from mnist_data_loader import load_mnist

SEED = 2026


In [ ]:
images_all, labels_all, images_test, labels_test = load_mnist()
assert images_all.shape == (60_000, 28, 28)
assert images_test.shape == (10_000, 28, 28)
X_all = images_all.reshape(60_000, 784).astype(np.float32) / 255.0
X_test = images_test.reshape(10_000, 784).astype(np.float32) / 255.0
y_all, y_test = labels_all, labels_test

# Identical split recipe to HW1: 50,000 training and 10,000 validation.
permutation = np.random.default_rng(SEED).permutation(60_000)
train_indices, val_indices = permutation[:50_000], permutation[50_000:]
X_train, y_train = X_all[train_indices], y_all[train_indices]
X_val, y_val = X_all[val_indices], y_all[val_indices]
del images_all, images_test, X_all
assert X_train.shape == (50_000, 784) and X_val.shape == (10_000, 784)
assert X_test.shape == (10_000, 784)
print('train / validation / test:', X_train.shape, X_val.shape, X_test.shape)


## 1. Model, objective, and function outputs

For a batch $X$ with shape $(B,784)$, use $W_1:(H,784)$, $b_1:(H,)$, $W_2:(10,H)$, and $b_2:(10,)$. The forward pass is $A=XW_1^\top+b_1$, $Q=\max(0,A)$, $Z=QW_2^\top+b_2$, and $P=\operatorname{softmax}(Z)$. Minimize mean cross-entropy plus $\lambda(\lVert W_1\rVert_F^2+\lVert W_2\rVert_F^2)/2$. The two biases are not regularized. Use ReLU derivative 0 at exactly 0.

| Function to fill | Required output |
|---|---|
| `init_params(D, H, C, seed)` | Dictionary with `W1` shape `(H,D)`, `b1` `(H,)`, `W2` `(C,H)`, `b2` `(C,)`; all NumPy arrays. |
| `forward(X, params)` | Tuple `(A, Q, Z, P)` with shapes `(B,H)`, `(B,H)`, `(B,C)`, `(B,C)`; each row of `P` sums to 1. |
| `loss_and_grad_batch(X, y, params, reg)` | `(loss, grads)`: finite Python `float` plus a dictionary with the same keys and array shapes as `params`. |
| `predict(X, params)` | Integer NumPy array of shape `(B,)`, with class labels 0–9. |
| `train_mlp(...)` | `(params, history)`; `history` has `train_loss`, `train_accuracy`, and `val_accuracy`, each a list of length `epochs`. |

The supplied `evaluate` helper returns `(loss, accuracy)`, both Python floats. For stable cross-entropy, use shifted logits and `log(sum(exp(shifted))) - shifted[y]`; directly taking `log(P[y])` can fail when a probability rounds to zero.


In [ ]:
def init_params(input_size, hidden_size, num_classes=10, seed=SEED):
    """Return {'W1':(H,D), 'b1':(H,), 'W2':(C,H), 'b2':(C,)} arrays.

    Use the supplied seed. Suggested Gaussian standard deviations are
    sqrt(2/D) for W1 and sqrt(1/H) for W2; initialize biases to zero.
    """
    rng = np.random.default_rng(seed)
    # TODO 1: Return the parameter dictionary with the stated shapes.
    raise NotImplementedError('Complete TODO 1: parameter initialization')


In [ ]:
def forward(X, params):
    """Return (A, Q, Z, P) with shapes (B,H), (B,H), (B,C), (B,C).

    X has shape (B,D). P must contain nonnegative probabilities whose
    rows sum to one. Use stable, row-wise softmax.
    """
    # TODO 2: Compute hidden preactivations, ReLU activations, logits,
    # and probabilities with NumPy array operations.
    raise NotImplementedError('Complete TODO 2: forward pass')


In [ ]:
def loss_and_grad_batch(X, y, params, reg):
    """Return (loss: Python float, grads: dict of NumPy arrays).

    X:(B,D), y:(B,), grads[key].shape == params[key].shape for every key.
    Use stable mean cross-entropy plus 0.5*reg*(sum(W1**2)+sum(W2**2)).
    Add reg*W to weight gradients; do not regularize biases.
    """
    if X.shape[0] == 0:
        raise ValueError('A batch must be nonempty')
    # TODO 3: Compute stable loss and vectorized backpropagation.
    # Do not modify the probabilities returned by forward in place.
    raise NotImplementedError('Complete TODO 3: loss and gradients')


In [ ]:
def predict(X, params):
    """Return integer class labels as a NumPy array with shape (B,)."""
    # TODO 4: Select the largest output logit in every row.
    raise NotImplementedError('Complete TODO 4: prediction')


In [ ]:
def evaluate(X, y, params, reg=0.0, chunk_size=2048):
    """Return (regularized mean loss, accuracy), both Python floats."""
    if len(X) == 0:
        raise ValueError('The evaluation set must be nonempty')
    total_nll, correct = 0.0, 0
    for start in range(0, len(X), chunk_size):
        X_part = X[start:start + chunk_size]
        y_part = y[start:start + chunk_size]
        Z = forward(X_part, params)[2]
        shifted = Z - np.max(Z, axis=1, keepdims=True)
        total_nll += float(np.sum(
            np.log(np.sum(np.exp(shifted), axis=1))
            - shifted[np.arange(len(X_part)), y_part]))
        correct += int(np.sum(np.argmax(Z, axis=1) == y_part))
    penalty = 0.5 * reg * (
        np.sum(params['W1'] ** 2) + np.sum(params['W2'] ** 2))
    return float(total_nll / len(X) + penalty), correct / len(X)


## 2. Check your implementation

Run this after TODO 1–4. The tests check output shapes, extreme logits, prediction, and analytic gradients against central finite differences on a tiny model. Loops in this diagnostic checker are allowed.


In [ ]:
shape_params = init_params(5, 3, 4, seed=SEED)
assert {key: value.shape for key, value in shape_params.items()} == {
    'W1': (3, 5), 'b1': (3,), 'W2': (4, 3), 'b2': (4,)}
A, Q, Z, P = forward(np.zeros((2, 5), dtype=np.float32), shape_params)
assert (A.shape, Q.shape, Z.shape, P.shape) == ((2, 3), (2, 3), (2, 4), (2, 4))
assert np.allclose(P.sum(axis=1), 1.0) and np.all(P >= 0)

extreme = {'W1': np.array([[1.0]]), 'b1': np.array([0.0]),
           'W2': np.array([[1000.0], [-1000.0]]), 'b2': np.zeros(2)}
extreme_loss, _ = loss_and_grad_batch(
    np.array([[1.0]]), np.array([1]), extreme, 0.0)
assert np.isfinite(extreme_loss) and extreme_loss > 1000.0

check_rng = np.random.default_rng(7)
X_check = check_rng.normal(scale=0.2, size=(4, 5))
y_check = np.array([0, 2, 1, 2])
check_params = {
    'W1': check_rng.normal(scale=0.1, size=(3, 5)),
    'b1': np.array([0.5, 0.6, 0.7]),
    'W2': check_rng.normal(scale=0.1, size=(3, 3)),
    'b2': check_rng.normal(scale=0.1, size=3),
}
reg_check = 0.03
_, analytic = loss_and_grad_batch(X_check, y_check, check_params, reg_check)
assert np.array_equal(predict(X_check, check_params),
                      np.argmax(forward(X_check, check_params)[2], axis=1))

step = 1e-5
largest_error = 0.0
for name, values in check_params.items():
    assert analytic[name].shape == values.shape
    for index in np.ndindex(values.shape):
        original = values[index]
        values[index] = original + step
        plus = loss_and_grad_batch(X_check, y_check, check_params, reg_check)[0]
        values[index] = original - step
        minus = loss_and_grad_batch(X_check, y_check, check_params, reg_check)[0]
        values[index] = original
        numeric = (plus - minus) / (2 * step)
        largest_error = max(largest_error, abs(numeric - analytic[name][index]))
assert largest_error < 1e-6, largest_error
print('Largest absolute gradient error:', largest_error)
print('All checks passed.')


## 3. Mini-batch training

Only training images enter parameter updates. The supplied loop records full training loss and training/validation accuracy after every epoch; `history` must contain one value per epoch for each metric.


In [ ]:
def train_mlp(X_train, y_train, X_val, y_val, *, hidden_size,
              learning_rate, reg=1e-4, epochs=3, batch_size=256, seed=SEED):
    """Return (params, history).

    params is the trained parameter dictionary. history has the keys
    'train_loss', 'train_accuracy', 'val_accuracy'; each value is a list
    of Python floats with length epochs.
    """
    params = init_params(X_train.shape[1], hidden_size, 10, seed)
    rng = np.random.default_rng(seed)
    history = {'train_loss': [], 'train_accuracy': [], 'val_accuracy': []}
    for epoch in range(epochs):
        order = rng.permutation(len(X_train))
        for start in range(0, len(order), batch_size):
            indices = order[start:start + batch_size]
            _, grads = loss_and_grad_batch(
                X_train[indices], y_train[indices], params, reg)
            # TODO 5: Update all four parameter arrays with SGD.
            # The gradients above came from the same old parameters.
            raise NotImplementedError('Complete TODO 5: parameter update')
        train_loss, train_acc = evaluate(X_train, y_train, params, reg)
        _, val_acc = evaluate(X_val, y_val, params)
        history['train_loss'].append(train_loss)
        history['train_accuracy'].append(train_acc)
        history['val_accuracy'].append(val_acc)
        print(f'epoch {epoch + 1:2d}/{epochs}: loss={train_loss:.4f}, '
              f'train acc={train_acc:.4f}, val acc={val_acc:.4f}')
    return params, history


## 4. Compare hyperparameters

These three runs compare two learning rates at width 32 and two widths at learning rate 0.10. Keep the split and seed fixed. Choose the model using validation accuracy only; do not inspect the test set here.


In [ ]:
configs = [
    {'name': 'baseline', 'hidden_size': 32, 'learning_rate': 0.10, 'reg': 1e-4},
    {'name': 'smaller learning rate', 'hidden_size': 32, 'learning_rate': 0.03, 'reg': 1e-4},
    {'name': 'wider hidden layer', 'hidden_size': 64, 'learning_rate': 0.10, 'reg': 1e-4},
]
runs = {}
for config in configs:
    print('Configuration:', config['name'])
    params, history = train_mlp(
        X_train, y_train, X_val, y_val,
        hidden_size=config['hidden_size'],
        learning_rate=config['learning_rate'], reg=config['reg'],
        epochs=3, batch_size=256, seed=SEED)
    runs[config['name']] = {'config': config, 'params': params, 'history': history}

print('configuration             width      lr   lambda  train acc   val acc')
for name, run in runs.items():
    cfg, h = run['config'], run['history']
    print(f"{name:26s} {cfg['hidden_size']:5d} {cfg['learning_rate']:7.3g} "
          f"{cfg['reg']:8.2g} {h['train_accuracy'][-1]:10.4f} "
          f"{h['val_accuracy'][-1]:9.4f}")
best_name = max(runs, key=lambda name: runs[name]['history']['val_accuracy'][-1])
print('Selected by final validation accuracy:', best_name)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, run in runs.items():
    h = run['history']
    epoch_axis = np.arange(1, len(h['train_loss']) + 1)
    axes[0].plot(epoch_axis, h['train_loss'], marker='o', label=name)
    axes[1].plot(epoch_axis, h['train_accuracy'], marker='o', label=f'{name}: train')
    axes[1].plot(epoch_axis, h['val_accuracy'], marker='x', linestyle='--',
                 label=f'{name}: val')
axes[0].set(xlabel='Epoch', ylabel='Training loss', title='Loss')
axes[1].set(xlabel='Epoch', ylabel='Accuracy', title='Accuracy')
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


## 5. One-time test evaluation

Run this only after selecting the configuration by validation accuracy. Do not change settings based on the test result.


In [ ]:
chosen = runs[best_name]
_, test_accuracy = evaluate(X_test, y_test, chosen['params'])
print('Selected configuration:', best_name)
print(f"Final validation accuracy: {chosen['history']['val_accuracy'][-1]:.4f}")
print(f'Test accuracy: {test_accuracy:.4f}')


## Report checklist

- Derive all four gradients, give their shapes, and explain the stable softmax calculation.
- Include the full 50,000/10,000/10,000 data sizes, settings table, and loss/accuracy curves.
- Compare learning rates and widths, discuss fit, and explain one difference from HW1's linear model.
- State the validation-selected configuration and its one-time test accuracy.
- Submit the executed notebook, `mnist_data_loader.py`, and a PDF report with your name and student ID. Do not submit `mnist_data/`.
